In [42]:
# Core data handling
import numpy as np
import pandas as pd
# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
# Modeling
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv('Historical Product Demand.csv')
df.shape

## Phase 1: Clean the data
### Fix 1 — Order_Demand has accounting-style negative notation, e.g. `(1000)` means `-1000`
Flag it **before** touching the column with any numeric conversion, since that destroys the parentheses info.

In [ ]:
# Step 1: flag rows with parentheses BEFORE any conversion happens
is_negative = df['Order_Demand'].astype(str).str.contains(r'\(', regex=True)
print('Rows with parentheses:', is_negative.sum())

In [ ]:
# Step 2: strip the parentheses characters (still just strings at this point)
cleaned = (
    df['Order_Demand']
    .astype(str)
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)

In [ ]:
# Step 3: NOW convert to numeric (only after brackets are gone)
cleaned = pd.to_numeric(cleaned, errors='coerce')

# check what actually fails to convert -- this is the REAL diagnostic
# (before, this was being run on already-corrupted data, which is why it only showed [nan])
bad_mask = cleaned.isna() & df['Order_Demand'].notna()
print('Rows that fail to convert even after stripping brackets:', bad_mask.sum())
print(df.loc[bad_mask, 'Order_Demand'].unique()[:30])

In [ ]:
# Step 4: flip the sign for rows that were originally in parentheses
cleaned[is_negative] = cleaned[is_negative] * -1

df['Order_Demand'] = cleaned

print('New min:', df['Order_Demand'].min())
print('New max:', df['Order_Demand'].max())
print('NaNs in Order_Demand:', df['Order_Demand'].isna().sum())

**Check the output above.** `New min` should now actually be negative (assuming real negative rows exist) — if it's still 0, the flag/flip logic didn't work and needs re-checking. Any `NaN`s here are genuine unparseable values (not the earlier false alarm) and should be inspected before deciding to drop them.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%Y/%m/%d')
df.info()

In [ ]:
df.describe()

### Fix 2 — Duplicates: check the grain before dropping anything
Compare **full-row duplicates** vs **duplicates on (Product_Code, Warehouse, Date) only, ignoring Order_Demand**.
- If the two counts are similar → likely true copy-paste duplicates, safe to drop.
- If subset duplicates are far more numerous → these are genuinely separate orders for the same product/warehouse/day, and should be **summed**, not dropped.

In [ ]:
full_row_dupes = df.duplicated().sum()
subset_dupes = df.duplicated(subset=['Product_Code', 'Warehouse', 'Date']).sum()

print('Full-row duplicates:', full_row_dupes)
print('Subset (Product, Warehouse, Date) duplicates:', subset_dupes)

**Decision:** rather than dropping duplicates, fix the grain properly — collapse the data to one row per (Product_Code, Warehouse, Product_Category, Date) by **summing** Order_Demand. This correctly handles both cases at once: true copy-paste duplicates get summed into one (harmless if they were identical), and genuine same-day multiple orders get correctly combined into a true daily total.

In [ ]:
df = (
    df.groupby(['Product_Code', 'Warehouse', 'Product_Category', 'Date'], as_index=False, dropna=False)
    ['Order_Demand'].sum()
)

print('Shape after fixing grain:', df.shape)
print('Remaining full-row duplicates:', df.duplicated().sum())

### Fix 3 — Missing dates: check concentration before dropping
Earlier finding: all 11,239 missing-date rows belong to a single warehouse, `Whse_A`, which is ~7.3% of that warehouse's rows and ~1% of the overall dataset.

In [ ]:
print('Missing dates:', df['Date'].isna().sum())
print(df[df['Date'].isna()].groupby('Warehouse').size())

**Decision:** 11,239 rows (~7.3% of Whse_A's rows, ~1% of the overall dataset) are missing dates and cannot be recovered from any other column. Since this project forecasts demand at the product-category level rather than comparing warehouses, these rows are dropped. Noted limitation: Whse_A's true demand history may be slightly undercounted as a result.

In [ ]:
df = df.dropna(subset=['Date']).reset_index(drop=True)
print('Final shape after Phase 1 cleaning:', df.shape)
df.isnull().sum()

In [ ]:
df.describe()

**Phase 1 complete.** `df` is now: grain-correct (one row per product/warehouse/date), sign-correct (negative demand properly negative), and free of unrecoverable missing dates. Ready for Phase 2 (granularity decisions — product vs. category level, weekly vs. daily aggregation).